In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.utils.data as data_utils
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import os
import random

from dataclasses import dataclass, field
from torchmetrics.classification import MulticlassAccuracy

from dnnhelper import EarlyStopping, Experiment

In [4]:
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [5]:
# load mnist dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [6]:
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

In [7]:
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [14]:
train_dl = data_utils.DataLoader(mnist_trainset, batch_size=64, shuffle=True)
test_dl = data_utils.DataLoader(mnist_testset, batch_size=64, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Define the layers of the CNN
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=(3,3), stride=2, padding=1)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Apply the first convolutional layer, followed by ReLU and max pooling
        x = self.pool(F.relu(self.conv1(x)))

        # Apply the second convolutional layer, followed by ReLU and max pooling
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten the output from the convolutional layers
        x = self.flatten(x)

        # Apply the first fully connected layer, followed by ReLU and dropout
        x = self.dropout(F.relu(self.fc1(x)))

        # Apply the second fully connected layer to get the final output
        x = self.fc2(x)

        return x

In [ ]:
experiments = []
val_loss_values_baseline = []
train_loss_values_baseline = []

baseline_exp = Experiment(
  name = "baseline",
  checkpoints_folder = 'models',
  checkpoint_name = "baseline.pt",
  model = SimpleCNN,
  use_early_stopping = True,
  loss_fn = nn.MSELoss,
  optimizer= torch.optim.Adam,
  lr = 1e-3,
  epochs = 10,
  epoch_count =  10,
  early_stopping= EarlyStopping,
  patience= 5,
  val_loss_values = val_loss_values_baseline,
  train_loss_values = train_loss_values_baseline,
  color = "#FF7F0E",
  alpha = .3,
)

experiments.append(baseline_exp)

In [ ]:
def fit(exp:Experiment, trainloader, valloader):
  exp.epoch_count = []
  exp.train_loss_values = []
  exp.val_loss_values = []
  if exp.use_early_stopping:
    early_stopping = EarlyStopping(exp.checkpoint_save_path, patience=exp.patience, min_delta=exp.min_delta)
  for epoch in range(exp.epochs):
    exp.model.train()
    loss_epoch = 0
    for i, data in enumerate(trainloader, 0):
      X = data[0]
      y = data[1]
      y_pred = exp.model(X)
      loss = exp.loss_fn(y_pred.squeeze(-1), y)

      loss_epoch += loss

      exp.optimizer.zero_grad()

      loss.backward()

      exp.optimizer.step()
    loss_val = 0
    exp.model.eval()
    for j, data in enumerate(valloader, 0):

      X = data[0]
      y = data[1]

      with torch.no_grad():

        y_pred = exp.model(X)
        loss = exp.loss_fn(y_pred.squeeze(-1), y)

      loss_val += loss
    exp.epoch_count.append(epoch)
    exp.train_loss_values.append(loss_epoch.detach().numpy()/len(trainloader))
    exp.val_loss_values.append(loss_val.detach().numpy()/len(valloader))


    print(f"Epoca: {epoch} |  Train Loss: {loss_epoch/len(trainloader)} | Val Loss: {loss_val/len(valloader)} ")

    if exp.use_early_stopping:
      early_stopping(loss_val/len(valloader), exp.model)
      if early_stopping.early_stop:
        print("Early stopping all'epoca:", epoch)
        break
      exp.model.load_state_dict(torch.load(exp.checkpoint_save_path))

In [ ]:
def evaluate(exp:Experiment, testloader):
  tot_loss = 0
  exp.model.eval()
  for j, data in enumerate(testloader, 0):

    X = data[0]
    y = data[1]

    with torch.no_grad():

      y_pred = exp.model(X)
      loss = exp.loss_fn(y_pred, y)

    tot_loss += loss
  return tot_loss.detach().numpy()/len(testloader)

In [ ]:
# Metric definitions

#Accuracy
accuracy_fn_train = MulticlassAccuracy(num_classes=10, average='micro')
accuracy_fn_test = MulticlassAccuracy(num_classes=10, average='micro')